In [1]:
# Data collection
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd
import numpy as np


#Load the data
data= gutenberg.raw('shakespeare-macbeth.txt')
# Save to a text file
with open('hamlet.txt', 'w') as file:
    file.write(data)

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\Amit\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\gutenberg.zip.


In [5]:
# Data preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# load the dataset
with open('hamlet.txt', 'r') as file:
    text = file.read().lower()

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1
total_words

3553

In [6]:
# Create input sequences
input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [7]:
input_sequences

[[1, 885],
 [1, 885, 4],
 [1, 885, 4, 41],
 [1, 885, 4, 41, 57],
 [1, 885, 4, 41, 57, 1388],
 [1, 885, 4, 41, 57, 1388, 1389],
 [1, 885, 4, 41, 57, 1388, 1389, 1390],
 [418, 1391],
 [418, 1391, 1392],
 [418, 1391, 1392, 419],
 [270, 2],
 [270, 2, 886],
 [270, 2, 886, 33],
 [270, 2, 886, 33, 196],
 [270, 2, 886, 33, 196, 298],
 [81, 72],
 [81, 72, 38],
 [81, 72, 38, 32],
 [81, 72, 38, 32, 196],
 [81, 72, 38, 32, 196, 336],
 [81, 72, 38, 32, 196, 336, 131],
 [10, 270],
 [10, 270, 886],
 [10, 270, 886, 80],
 [10, 270, 886, 80, 10],
 [10, 270, 886, 80, 10, 1393],
 [128, 72],
 [128, 72, 1],
 [128, 72, 1, 1394],
 [128, 72, 1, 1394, 1395],
 [128, 72, 1, 1394, 1395, 84],
 [72, 1],
 [72, 1, 1396],
 [72, 1, 1396, 365],
 [72, 1, 1396, 365, 2],
 [72, 1, 1396, 365, 2, 887],
 [135, 7],
 [135, 7, 36],
 [135, 7, 36, 16],
 [135, 7, 36, 16, 172],
 [135, 7, 36, 16, 172, 1],
 [135, 7, 36, 16, 172, 1, 299],
 [135, 7, 36, 16, 172, 1, 299, 4],
 [135, 7, 36, 16, 172, 1, 299, 4, 666],
 [81, 76],
 [81, 76, 1],


In [8]:
# pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
max_sequence_len

14

In [9]:
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
input_sequences

array([[   0,    0,    0, ...,    0,    1,  885],
       [   0,    0,    0, ...,    1,  885,    4],
       [   0,    0,    0, ...,  885,    4,   41],
       ...,
       [   0,    0,    0, ..., 3552,    1,  885],
       [   0,    0,    0, ...,    1,  885,    4],
       [   0,    0,    0, ...,  885,    4,   41]])

In [10]:
# Create predictors and label
import tensorflow as tf
x = input_sequences[:,:-1]
y = input_sequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [11]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [12]:
# split the data
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [14]:
# Training the model LSTM RNN 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Build the model
model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 13, 100)           355300    
                                                                 
 lstm_2 (LSTM)               (None, 13, 150)           150600    
                                                                 
 dropout_1 (Dropout)         (None, 13, 150)           0         
                                                                 
 lstm_3 (LSTM)               (None, 100)               100400    
                                                                 
 dense_1 (Dense)             (None, 3553)              358853    
                                                                 
Total params: 965153 (3.68 MB)
Trainable params: 965153 (3.68 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [16]:
# Training the model
history = model.fit(X_train, y_train, epochs=100, verbose=1, validation_data=(X_test, y_test))

Epoch 1/100
382/382 [==============================] - 4s 11ms/step - loss: 2.8535 - accuracy: 0.3887 - val_loss: 11.2390 - val_accuracy: 0.0492
Epoch 2/100
382/382 [==============================] - 4s 11ms/step - loss: 2.8179 - accuracy: 0.3956 - val_loss: 11.3060 - val_accuracy: 0.0499
Epoch 3/100
382/382 [==============================] - 4s 11ms/step - loss: 2.7751 - accuracy: 0.4019 - val_loss: 11.3764 - val_accuracy: 0.0495
Epoch 4/100
382/382 [==============================] - 4s 11ms/step - loss: 2.7449 - accuracy: 0.4075 - val_loss: 11.4552 - val_accuracy: 0.0502
Epoch 5/100
382/382 [==============================] - 4s 11ms/step - loss: 2.7031 - accuracy: 0.4179 - val_loss: 11.5544 - val_accuracy: 0.0479
Epoch 6/100
382/382 [==============================] - 4s 11ms/step - loss: 2.6711 - accuracy: 0.4216 - val_loss: 11.5991 - val_accuracy: 0.0508
Epoch 7/100
382/382 [==============================] - 4s 11ms/step - loss: 2.6381 - accuracy: 0.4259 - val_loss: 11.6412 - val_ac

In [17]:
# Fucmtipm to predict the next word
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) > max_sequence_len - 1:
        token_list = token_list[-(max_sequence_len - 1):]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)[0]
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None

In [18]:
input_text = "to be or not to be"
print(f"Input Text: {input_text}")
max_sequence_len = model.input_shape[1] + 1
next_word = predict_next_word(model, tokenizer, input_text, max_sequence_len)
print(f"Predicted Next Word: {next_word}")

Input Text: to be or not to be
Predicted Next Word: done


In [19]:
# Save the model
model.save('next_word_prediction_model.h5')
# save the tokenizer
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

c:\Projects\RNN_nextwordpred\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
